#  <center>Taller de Aprendizaje Automatico</center>
##  <center>Paso 3: Procesamiento de datos</center>
### <center>Procesamiento base, intento 2 para arboles</center>

Este notebook toma el flujo base de procesamiento y le agrega una variante pensada para modelos de arboles: conservar las features originales sin escalar, sin imputar y sin descartar columnas por missing, pero reemplazando el marcador `-999` por `NaN` para explicitar mejor los valores faltantes.

La idea es que este intento quede como una alternativa simple y trazable. Primero se cargan los datos, despues se separan identificadores, target y features, y por ultimo se aplican dos decisiones acotadas: convertir `-999` en `NaN` y hacer one-hot encoding de `PRI_jet_num`, manteniendo intacto el resto del dataset.

## Carga reproducible del dataset

Se cargan `training.csv` y `test.csv` desde `data/higgs-boson/`. El archivo de entrenamiento contiene `Label` y `Weight`; el archivo de test no.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


def replace_missing_marker_with_nan(
    data: pd.DataFrame,
    missing_marker: float = -999,
) -> pd.DataFrame:
    return data.replace(missing_marker, np.nan)


project_root = find_project_root()
data_dir = project_root / "data" / "higgs-boson"
train_path = data_dir / "training.csv"
test_path = data_dir / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

if not test_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print(f"project_root = {project_root}")
print(f"train shape = {train_data.shape}")
print(f"test shape = {test_data.shape}")


project_root = /home/santig14/fing/taa/1_TallerAA
train shape = (250000, 33)
test shape = (550000, 31)


## Verificacion visual de la carga

Se muestran las primeras filas de training y test para dejar evidencia de que la carga fue correcta.

In [2]:
display(train_data.head())
display(test_data.head())


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,350000,-999.000,79.589,23.916,3.036,-999.000,-999.000,-999.000,0.903,3.036,...,2.022,98.556,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,-0.000
1,350001,106.398,67.490,87.949,49.994,-999.000,-999.000,-999.000,2.048,2.679,...,-1.138,176.251,1,47.575,-0.553,-0.849,-999.000,-999.000,-999.000,47.575
2,350002,117.794,56.226,96.358,4.137,-999.000,-999.000,-999.000,2.755,4.137,...,-1.868,111.505,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
3,350003,135.861,30.604,97.288,9.104,-999.000,-999.000,-999.000,2.811,9.104,...,1.172,164.707,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
4,350004,74.159,82.772,58.731,89.646,1.347,536.663,-0.339,1.028,77.213,...,-0.231,869.614,3,254.085,-1.013,-0.334,185.857,0.335,2.587,599.213


## Separacion base para entrenamiento

Antes de cualquier transformacion, se separan las columnas que no deben entrar como features del modelo:

- `EventId`: identificador del evento.
- `Weight`: peso asociado al evento, que no se usa como feature.
- `Label`: variable objetivo, que pasa a `y_train`.

Con eso se arman `X_train`, `y_train` y `X_test` como punto de partida comun.

In [3]:
excluded_train_columns = ["EventId", "Weight", "Label"]
feature_columns = [
    column for column in train_data.columns
    if column not in excluded_train_columns
]

train_event_id = train_data["EventId"].copy()
train_weight = train_data["Weight"].copy()
y_train = train_data["Label"].copy()
X_train = train_data[feature_columns].copy()

test_event_id = test_data["EventId"].copy()
X_test = test_data[feature_columns].copy()

expected_test_columns = [column for column in test_data.columns if column != "EventId"]
if feature_columns != expected_test_columns:
    raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

print(f"Cantidad de features iniciales: {len(feature_columns)}")
print(f"X_train shape = {X_train.shape}")
print(f"y_train shape = {y_train.shape}")
print(f"X_test shape = {X_test.shape}")


Cantidad de features iniciales: 30
X_train shape = (250000, 30)
y_train shape = (250000,)
X_test shape = (550000, 30)


## Regla del intento 2

En este intento no se elimina ninguna feature por presencia de `-999`, no se imputan esos valores y tampoco se reescala ninguna columna. La unica limpieza explicita consiste en reemplazar el marcador `-999` por `NaN`, para distinguir mejor los faltantes del resto de los valores numericos. Sobre esa base, se aplica solo la codificacion one-hot de `PRI_jet_num`.

In [4]:
missing_marker = -999

marker_rows = []
for column in X_train.columns:
    train_marker_count = int((X_train[column] == missing_marker).sum())
    train_marker_pct = round(train_marker_count / len(X_train) * 100, 2)
    marker_rows.append(
        {
            "feature": column,
            "train_minus_999_count": train_marker_count,
            "train_minus_999_pct": train_marker_pct,
        }
    )

marker_summary_i2 = pd.DataFrame(marker_rows).sort_values(
    ["train_minus_999_pct", "feature"],
    ascending=[False, True],
)

display(marker_summary_i2.head(10).reset_index(drop=True))
print("Columnas eliminadas en intento 2: 0")
print("En intento 2 no se descarta ninguna feature por presencia de -999.")


,feature,train_minus_999_count,train_minus_999_pct
0,DER_deltaeta_jet_jet,177457,70.98
1,DER_lep_eta_centrality,177457,70.98
2,DER_mass_jet_jet,177457,70.98
3,DER_prodeta_jet_jet,177457,70.98
4,PRI_jet_subleading_eta,177457,70.98
5,PRI_jet_subleading_phi,177457,70.98
6,PRI_jet_subleading_pt,177457,70.98
7,PRI_jet_leading_eta,99913,39.97
8,PRI_jet_leading_phi,99913,39.97
9,PRI_jet_leading_pt,99913,39.97


Columnas eliminadas en intento 2: 0
En intento 2 no se descarta ninguna feature por presencia de -999.


## Conservacion de las features originales

Como en este intento no se descarta ninguna columna, `X_train_i2` y `X_test_i2` conservan las `30` features originales del dataset. A partir de este punto, ambas tablas siguen intactas hasta el one-hot encoding de `PRI_jet_num`.

In [5]:
X_train_i2 = X_train.copy()
X_test_i2 = X_test.copy()
feature_columns_i2 = X_train_i2.columns.tolist()

print(f"X_train_i2 shape = {X_train_i2.shape}")
print(f"X_test_i2 shape = {X_test_i2.shape}")
print(f"Cantidad de features conservadas en intento 2: {len(feature_columns_i2)}")

display(pd.DataFrame({"kept_feature": feature_columns_i2}).head(10))


X_train_i2 shape = (250000, 30)
X_test_i2 shape = (550000, 30)
Cantidad de features conservadas en intento 2: 30


,kept_feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltaeta_jet_jet
5,DER_mass_jet_jet
6,DER_prodeta_jet_jet
7,DER_deltar_tau_lep
8,DER_pt_tot
9,DER_sum_pt


## Reemplazo simple de `-999` por `NaN`

En este intento los valores `-999` no se usan para descartar columnas ni se imputan despues. Solo se reemplazan por `NaN` para que el dataset exprese los faltantes de forma estandar antes de seguir con el one-hot encoding.

In [6]:
remaining_marker_count_train = int((X_train_i2 == missing_marker).sum().sum())
remaining_marker_count_test = int((X_test_i2 == missing_marker).sum().sum())
total_cells_train_i2 = int(X_train_i2.shape[0] * X_train_i2.shape[1])
total_cells_test_i2 = int(X_test_i2.shape[0] * X_test_i2.shape[1])
remaining_marker_pct_train = remaining_marker_count_train / total_cells_train_i2 * 100
remaining_marker_pct_test = remaining_marker_count_test / total_cells_test_i2 * 100

X_train_i2_nan = replace_missing_marker_with_nan(X_train_i2, missing_marker=missing_marker)
X_test_i2_nan = replace_missing_marker_with_nan(X_test_i2, missing_marker=missing_marker)

remaining_nan_count_train = int(X_train_i2_nan.isna().sum().sum())
remaining_nan_count_test = int(X_test_i2_nan.isna().sum().sum())

print(
    f"Marcadores -999 presentes en X_train_i2 antes del reemplazo: {remaining_marker_count_train} "
    f"({remaining_marker_pct_train:.2f}% del total de celdas)"
)
print(
    f"Marcadores -999 presentes en X_test_i2 antes del reemplazo: {remaining_marker_count_test} "
    f"({remaining_marker_pct_test:.2f}% del total de celdas)"
)
print(f"NaN presentes en X_train_i2_nan despues del reemplazo: {remaining_nan_count_train}")
print(f"NaN presentes en X_test_i2_nan despues del reemplazo: {remaining_nan_count_test}")
print("No se realiza ninguna imputacion sobre estos NaN.")


Marcadores -999 presentes en X_train_i2 antes del reemplazo: 1580052 (21.07% del total de celdas)
Marcadores -999 presentes en X_test_i2 antes del reemplazo: 3473394 (21.05% del total de celdas)
NaN presentes en X_train_i2_nan despues del reemplazo: 1580052
NaN presentes en X_test_i2_nan despues del reemplazo: 3473394
No se realiza ninguna imputacion sobre estos NaN.


## One-hot encoding de `PRI_jet_num`

La columna `PRI_jet_num` representa una cantidad discreta de jets. En lugar de tratarla como una variable numerica continua, en este paso se transforma con `OneHotEncoder` para generar una columna binaria por categoria observada en training. La codificacion se ajusta solo con training y despues se replica sobre test. El resto de las columnas se conserva sin escalar y con los faltantes expresados como `NaN`.

In [7]:
jet_num_column = "PRI_jet_num"
remaining_numeric_columns_i2 = [column for column in feature_columns_i2 if column != jet_num_column]

jet_num_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_i2_jet_num = X_train_i2_nan[[jet_num_column]].astype(int)
X_test_i2_jet_num = X_test_i2_nan[[jet_num_column]].astype(int)

X_train_i2_jet_num_encoded_array = jet_num_encoder.fit_transform(X_train_i2_jet_num)
jet_num_encoded_columns = jet_num_encoder.get_feature_names_out([jet_num_column]).tolist()

X_train_i2_jet_num_encoded = pd.DataFrame(
    X_train_i2_jet_num_encoded_array,
    columns=jet_num_encoded_columns,
    index=X_train_i2_nan.index,
)

X_test_i2_jet_num_encoded = pd.DataFrame(
    jet_num_encoder.transform(X_test_i2_jet_num),
    columns=jet_num_encoded_columns,
    index=X_test_i2_nan.index,
)

print(f"Columnas generadas para {jet_num_column}: {jet_num_encoded_columns}")
print(f"X_train_i2_jet_num_encoded shape = {X_train_i2_jet_num_encoded.shape}")
print(f"X_test_i2_jet_num_encoded shape = {X_test_i2_jet_num_encoded.shape}")


Columnas generadas para PRI_jet_num: ['PRI_jet_num_0', 'PRI_jet_num_1', 'PRI_jet_num_2', 'PRI_jet_num_3']
X_train_i2_jet_num_encoded shape = (250000, 4)
X_test_i2_jet_num_encoded shape = (550000, 4)


## Armado final sin escalado ni imputacion

Despues de codificar `PRI_jet_num`, el resto de las columnas numericas se conserva en su escala original. Los faltantes quedan expresados como `NaN` y el dataset final se arma concatenando esas columnas sin escalar con las nuevas columnas binarias del one-hot encoding.

In [8]:
X_train_i2_passthrough_numeric = X_train_i2_nan[remaining_numeric_columns_i2].copy()
X_test_i2_passthrough_numeric = X_test_i2_nan[remaining_numeric_columns_i2].copy()

X_train_i2_final = pd.concat([X_train_i2_passthrough_numeric, X_train_i2_jet_num_encoded], axis=1)
X_test_i2_final = pd.concat([X_test_i2_passthrough_numeric, X_test_i2_jet_num_encoded], axis=1)
feature_columns_i2_final = X_train_i2_final.columns.tolist()

retained_nan_count_train = int(X_train_i2_final[remaining_numeric_columns_i2].isna().sum().sum())
retained_nan_count_test = int(X_test_i2_final[remaining_numeric_columns_i2].isna().sum().sum())
retained_marker_count_train = int((X_train_i2_final[remaining_numeric_columns_i2] == missing_marker).sum().sum())
retained_marker_count_test = int((X_test_i2_final[remaining_numeric_columns_i2] == missing_marker).sum().sum())

print(f"X_train_i2_final shape = {X_train_i2_final.shape}")
print(f"X_test_i2_final shape = {X_test_i2_final.shape}")
print(f"Cantidad final de features despues de one-hot encoding: {len(feature_columns_i2_final)}")
print(f"NaN conservados en X_train_i2_final: {retained_nan_count_train}")
print(f"NaN conservados en X_test_i2_final: {retained_nan_count_test}")
print(f"Marcadores -999 restantes en X_train_i2_final: {retained_marker_count_train}")
print(f"Marcadores -999 restantes en X_test_i2_final: {retained_marker_count_test}")

display(pd.DataFrame({"final_feature": feature_columns_i2_final}).head(10))


X_train_i2_final shape = (250000, 33)
X_test_i2_final shape = (550000, 33)
Cantidad final de features despues de one-hot encoding: 33
NaN conservados en X_train_i2_final: 1580052
NaN conservados en X_test_i2_final: 3473394
Marcadores -999 restantes en X_train_i2_final: 0
Marcadores -999 restantes en X_test_i2_final: 0


,final_feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltaeta_jet_jet
5,DER_mass_jet_jet
6,DER_prodeta_jet_jet
7,DER_deltar_tau_lep
8,DER_pt_tot
9,DER_sum_pt


## Objetos resultantes del intento 2

Se muestran las primeras filas de `X_train_i2_final` y de `y_train` para confirmar que la separacion sigue bien y que el intento 2 queda listo sin descarte de columnas, sin imputacion y sin escalado, dejando los faltantes como `NaN`.

In [9]:
display(X_train_i2_final.head())
display(y_train.head().to_frame(name="Label"))


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,2.150,0.444,46.062,1.24,-2.475,113.497,0.0,0.0,1.0,0.0
1,160.937,68.768,103.235,48.146,NaN,NaN,NaN,3.473,2.078,125.157,...,0.725,1.158,NaN,NaN,NaN,46.226,0.0,1.0,0.0,0.0
2,NaN,162.172,125.953,35.635,NaN,NaN,NaN,3.148,9.336,197.814,...,2.053,-2.028,NaN,NaN,NaN,44.251,0.0,1.0,0.0,0.0
3,143.905,81.417,80.943,0.414,NaN,NaN,NaN,3.310,0.414,75.968,...,NaN,NaN,NaN,NaN,NaN,-0.000,1.0,0.0,0.0,0.0
4,175.864,16.915,134.805,16.405,NaN,NaN,NaN,3.891,16.405,57.983,...,NaN,NaN,NaN,NaN,NaN,0.000,1.0,0.0,0.0,0.0


,Label
0,s
1,b
2,b
3,b
4,b


## Pipeline final reutilizable

A continuacion se deja una version reusable del flujo completo del intento 2. Esta version parte desde la carga de datos, separa columnas especiales, conserva todas las features, reemplaza `-999` por `NaN`, aplica solo `OneHotEncoder` sobre `PRI_jet_num` y devuelve los datasets transformados listos para entrenar modelos de arboles.

In [10]:
def load_higgs_data(project_root: Path | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    root = find_project_root(project_root)
    data_dir = root / "data" / "higgs-boson"
    train_path = data_dir / "training.csv"
    test_path = data_dir / "test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

    if not test_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

    return pd.read_csv(train_path), pd.read_csv(test_path)


def split_higgs_datasets(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series, pd.DataFrame, pd.DataFrame]:
    excluded_train_columns = ["EventId", "Weight", "Label"]
    feature_columns = [
        column for column in train_data.columns
        if column not in excluded_train_columns
    ]

    expected_test_columns = [column for column in test_data.columns if column != "EventId"]
    if feature_columns != expected_test_columns:
        raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

    train_event_id = train_data["EventId"].copy()
    train_weight = train_data["Weight"].copy()
    y_train = train_data["Label"].copy()
    X_train = train_data[feature_columns].copy()

    test_event_id = test_data["EventId"].copy()
    X_test = test_data[feature_columns].copy()

    return train_event_id, train_weight, y_train, test_event_id, X_train, X_test


def summarize_attempt2_marker_usage(
    X_train: pd.DataFrame,
    missing_marker: float = -999,
) -> pd.DataFrame:
    marker_summary = pd.DataFrame(
        {
            "feature": X_train.columns,
            "train_minus_999_count": [(X_train[column] == missing_marker).sum() for column in X_train.columns],
        }
    )
    marker_summary["train_minus_999_pct"] = (
        marker_summary["train_minus_999_count"] / len(X_train) * 100
    ).round(2)
    marker_summary = marker_summary.sort_values(
        ["train_minus_999_pct", "feature"],
        ascending=[False, True],
    )

    return marker_summary


def build_attempt2_preprocessing_pipeline(
    feature_columns: list[str],
    jet_num_column: str = "PRI_jet_num",
    missing_marker: float = -999,
) -> tuple[Pipeline, list[str], list[str]]:
    if jet_num_column not in feature_columns:
        raise ValueError(f"La columna {jet_num_column} debe existir en feature_columns.")

    numeric_columns = [column for column in feature_columns if column != jet_num_column]

    replace_numeric_missing = FunctionTransformer(
        replace_missing_marker_with_nan,
        validate=False,
        feature_names_out="one-to-one",
        kw_args={"missing_marker": missing_marker},
    )
    replace_jet_num_missing = FunctionTransformer(
        replace_missing_marker_with_nan,
        validate=False,
        feature_names_out="one-to-one",
        kw_args={"missing_marker": missing_marker},
    )

    numeric_pipeline = Pipeline(
        steps=[
            ("replace_missing_marker", replace_numeric_missing),
        ]
    )

    jet_num_pipeline = Pipeline(
        steps=[
            ("replace_missing_marker", replace_jet_num_missing),
            ("one_hot", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("jet_num", jet_num_pipeline, [jet_num_column]),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    preprocessing_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
        ]
    )

    return preprocessing_pipeline, numeric_columns, [jet_num_column]


def transform_attempt2_with_pipeline(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    missing_marker: float = -999,
) -> dict[str, object]:
    train_event_id, train_weight, y_train, test_event_id, X_train, X_test = split_higgs_datasets(
        train_data,
        test_data,
    )

    feature_columns_i2 = X_train.columns.tolist()
    drop_columns_i2: list[str] = []
    marker_summary_i2 = summarize_attempt2_marker_usage(
        X_train,
        missing_marker=missing_marker,
    )

    preprocessing_pipeline, numeric_columns_i2, categorical_columns_i2 = build_attempt2_preprocessing_pipeline(
        feature_columns_i2,
        missing_marker=missing_marker,
    )

    X_train_selected = X_train[feature_columns_i2].copy()
    X_test_selected = X_test[feature_columns_i2].copy()

    X_train_processed_array = preprocessing_pipeline.fit_transform(X_train_selected)
    X_test_processed_array = preprocessing_pipeline.transform(X_test_selected)

    processed_feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out().tolist()

    X_train_processed = pd.DataFrame(
        X_train_processed_array,
        columns=processed_feature_names,
        index=X_train_selected.index,
    )

    X_test_processed = pd.DataFrame(
        X_test_processed_array,
        columns=processed_feature_names,
        index=X_test_selected.index,
    )

    return {
        "train_event_id": train_event_id,
        "train_weight": train_weight,
        "y_train": y_train,
        "test_event_id": test_event_id,
        "feature_columns_i2": feature_columns_i2,
        "drop_columns_i2": drop_columns_i2,
        "marker_summary_i2": marker_summary_i2,
        "numeric_columns_i2": numeric_columns_i2,
        "categorical_columns_i2": categorical_columns_i2,
        "processed_feature_names": processed_feature_names,
        "preprocessing_pipeline": preprocessing_pipeline,
        "X_train_processed": X_train_processed,
        "X_test_processed": X_test_processed,
    }


train_data_pipeline, test_data_pipeline = load_higgs_data()
attempt2_pipeline_artifacts = transform_attempt2_with_pipeline(train_data_pipeline, test_data_pipeline)

print(f"Columnas descartadas en intento 2: {attempt2_pipeline_artifacts['drop_columns_i2']}")
print(f"Cantidad de features conservadas antes del pipeline: {len(attempt2_pipeline_artifacts['feature_columns_i2'])}")
print(f"Cantidad final de features procesadas: {len(attempt2_pipeline_artifacts['processed_feature_names'])}")
print(f"X_train_processed shape = {attempt2_pipeline_artifacts['X_train_processed'].shape}")
print(f"X_test_processed shape = {attempt2_pipeline_artifacts['X_test_processed'].shape}")
print(
    f"NaN en X_train_processed: {int(attempt2_pipeline_artifacts['X_train_processed'].isna().sum().sum())}"
)
print(
    f"NaN en X_test_processed: {int(attempt2_pipeline_artifacts['X_test_processed'].isna().sum().sum())}"
)

display(attempt2_pipeline_artifacts['X_train_processed'].head())
display(attempt2_pipeline_artifacts['y_train'].head().to_frame(name='Label'))


Columnas descartadas en intento 2: []
Cantidad de features conservadas antes del pipeline: 30
Cantidad final de features procesadas: 33
X_train_processed shape = (250000, 33)
X_test_processed shape = (550000, 33)
NaN en X_train_processed: 1580052
NaN en X_test_processed: 3473394


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,2.150,0.444,46.062,1.24,-2.475,113.497,0.0,0.0,1.0,0.0
1,160.937,68.768,103.235,48.146,NaN,NaN,NaN,3.473,2.078,125.157,...,0.725,1.158,NaN,NaN,NaN,46.226,0.0,1.0,0.0,0.0
2,NaN,162.172,125.953,35.635,NaN,NaN,NaN,3.148,9.336,197.814,...,2.053,-2.028,NaN,NaN,NaN,44.251,0.0,1.0,0.0,0.0
3,143.905,81.417,80.943,0.414,NaN,NaN,NaN,3.310,0.414,75.968,...,NaN,NaN,NaN,NaN,NaN,-0.000,1.0,0.0,0.0,0.0
4,175.864,16.915,134.805,16.405,NaN,NaN,NaN,3.891,16.405,57.983,...,NaN,NaN,NaN,NaN,NaN,0.000,1.0,0.0,0.0,0.0


,Label
0,s
1,b
2,b
3,b
4,b


## Guardado de los datos procesados

Para cerrar el intento 2, se guardan los datasets procesados y los artefactos principales en `data/higgs-boson/processed/i2_arboles/`. Esto deja disponible una salida reutilizable para los siguientes pasos de modelado sin tener que reejecutar manualmente todas las transformaciones.

In [11]:
def save_attempt2_processed_data(
    artifacts: dict[str, object],
    output_dir: Path,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    files_to_save: dict[str, pd.DataFrame] = {
        "X_train_processed.csv": artifacts["X_train_processed"],
        "X_test_processed.csv": artifacts["X_test_processed"],
        "y_train.csv": artifacts["y_train"].to_frame(name="Label"),
        "train_weight.csv": artifacts["train_weight"].to_frame(name="Weight"),
        "train_event_id.csv": artifacts["train_event_id"].to_frame(name="EventId"),
        "test_event_id.csv": artifacts["test_event_id"].to_frame(name="EventId"),
        "drop_columns_i2.csv": pd.DataFrame({"dropped_feature": artifacts["drop_columns_i2"]}),
        "feature_columns_i2.csv": pd.DataFrame({"feature": artifacts["feature_columns_i2"]}),
        "processed_feature_names.csv": pd.DataFrame({"feature": artifacts["processed_feature_names"]}),
        "marker_summary_i2.csv": artifacts["marker_summary_i2"],
    }

    saved_paths: list[Path] = []
    for filename, dataframe in files_to_save.items():
        file_path = output_dir / filename
        dataframe.to_csv(file_path, index=False)
        saved_paths.append(file_path)

    return saved_paths


processed_output_dir = find_project_root() / "data" / "higgs-boson" / "processed" / "i2_arboles"
saved_paths_i2 = save_attempt2_processed_data(attempt2_pipeline_artifacts, processed_output_dir)

print(f"Directorio de salida: {processed_output_dir}")
print(f"Cantidad de archivos guardados: {len(saved_paths_i2)}")
for path in saved_paths_i2:
    print(f"- {path}")


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles
Cantidad de archivos guardados: 10
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/test_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/drop_columns_i2.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/feature_columns_i2.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i2_arboles/processed_feature_names.csv
- /home/santig1